# Google Colab · end-to-end banking analytics
Upload a zip of this repository (without real customer data) first, or clone your own repository into `/content/banking-customer-analytics`. No public repository URL is assumed.

In [ ]:
from pathlib import Path
import os, sys
PROJECT = Path('/content/banking-customer-analytics')
if not PROJECT.exists():
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    archive = next((name for name in uploaded if name.endswith('.zip')), None)
    if archive is None: raise ValueError('Upload the repository ZIP.')
    with zipfile.ZipFile(archive) as z:
        destination = Path('/content').resolve()
        for member in z.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination):
                raise ValueError('Unsafe ZIP path')
        z.extractall(destination)
if not (PROJECT / 'requirements.txt').exists():
    raise FileNotFoundError('Set PROJECT to the extracted repository directory.')
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

In [ ]:
%pip install -q -r requirements.txt

Optionally upload Excel. Set the sheet and zero-based header row to the actual file layout. The supplied dataset uses `Total Data`, header row 1. Files with other schemas fail explicitly.

In [ ]:
UPLOAD_EXCEL = True
if UPLOAD_EXCEL:
    from google.colab import files
    uploaded = files.upload()
    excel = [name for name in uploaded if name.lower().endswith('.xlsx')]
    if len(excel) != 1: raise ValueError('Upload exactly one Excel file.')
    import shutil, yaml
    destination = PROJECT / 'data/raw/customers.xlsx'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(excel[0], destination)
    path = PROJECT / 'config/config.yaml'
    settings = yaml.safe_load(path.read_text())
    settings['paths']['source'] = 'data/raw/customers.xlsx'
    path.write_text(yaml.safe_dump(settings, allow_unicode=True, sort_keys=False))

In [ ]:
from src.pipeline import run
from src.utils.config import load_config, location
import pandas as pd
config = load_config()
batch = run()
gold = location(config, 'gold') / batch
print('CSV and Parquet exports:', gold)
display(pd.read_csv(location(config, 'reports') / batch / 'data_quality_report.csv'))
display(pd.read_parquet(gold / 'gold_customer_kpis.parquet').T)

In [ ]:
# Download exports only when authorized to move customer data off this environment.
DOWNLOAD_EXPORTS = False
if DOWNLOAD_EXPORTS:
    import shutil
    from google.colab import files
    archive = shutil.make_archive('/content/gold_exports', 'zip', gold)
    files.download(archive)